## Make the 10kb, CTCF-containing boundaries used for pileups in Fig. 2 and comparisons in supp. Fig XX

In [68]:
import pandas as pd
import os
import bioframe
import pybedtools as pbt 
import numpy as np

In [69]:
basedir = '/mnt/md0/varshini/Analysis/Blood/complete_analyses/boundaries/'
savedir = '/mnt/md0/varshini/Analysis/Blood/complete_analyses/track_defs/erythroid_tracks/classifications/'
ctcf_bounds = pd.read_csv('/mnt/md0/varshini/Analysis/Blood/complete_analyses/track_defs/merged_CTCF_P123_strand.bed',
                         names=['chrom','start','end','CTCF','score','strand'], sep='\t').sort_values(by=['chrom','start'])


In [58]:
phases = ['Phase1_','Phase2_dedup', 'Phase3_dedup']

# create TAD boundaries from insulation score 
ins_res = 10_000
insulation_filenames = [os.path.join(basedir, f"{p}_microc_insulation_{ins_res}bp.tsv") for p in phases]
insulation_files = [pd.read_csv(i, sep='\t') for i in insulation_filenames]

w = 100_000

In [60]:
# make boundaries for each phase 
bounds_tables = []
ctcf_bed = pbt.BedTool.from_dataframe(ctcf_bounds)

for i in range(3):
    insulation_table = insulation_files[i]
    bounds = insulation_table[insulation_table[f'is_boundary_{w}'] == True][['chrom','start','end']]
    bounds_bed = pbt.BedTool.from_dataframe(bounds)
    
    bounds_with_ctcf = bounds_bed.intersect(ctcf_bed).to_dataframe()
    bounds_tables.append(bounds_with_ctcf)
    bounds_with_ctcf[['chrom','start','end']].drop_duplicates().to_csv(
        os.path.join(savedir, f"P{i+1}_CTCF_10kb_boundaries_{w//1000}kbw.bed"),
                       header=False,
                       index=False,
                       sep="\t")

In [61]:
# merged CTCF boundaries across phases
bounds_all = pd.concat([bounds_tables[0], bounds_tables[1], bounds_tables[2]], axis=0).drop_duplicates()
bounds_all.to_csv(os.path.join(savedir, f"CTCF_10kb_boundaries_{w//1000}kbw.bed"),
                       header=False,
                       index=False,
                       sep="\t")